# IBNN vs No-IBNN vs Fuzzy-GCD on WireGuard Flow Traffic

This notebook implements three controlled experiments.

1. IBNN following the architecture and information-bottleneck objective of Lin and Chen.
2. No-IBNN using the same convolutional extractor, concatenation, fully connected layers, optimizer, schedule, split and preprocessing, with only the stochastic IB layer and MI loss removed.
3. Fuzzy-GCD using the same No-IBNN architecture and parameter dimensions, with four existing encrypted-flow feature slots replaced by four packet-size structural features.

The original paper specifies four convolutional layers with 5x5, 5x5, 3x3 and 3x3 kernels and 32, 32, 64 and 64 channels, followed by concatenation, a 128-neuron feature extraction layer, the IB layer, 128- and 256-neuron fully connected layers and the output layer. Its tractable IB objective is cross-entropy plus beta times the estimated I(X;T), with L2 regularization.


In [2]:
!pip install -q pyarrow fastparquet scikit-learn pandas numpy matplotlib tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 7.4 MB/s eta 0:00:00a 0:00:01


In [3]:
import os
import ast
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)


cpu


## Dataset

The dataset contains two session Parquet files. Each flow has 126 columns. The encrypted-side block contains 32 derived numeric features and the dataset also provides the first 255 encrypted packet sizes in `outer_splt_ps`.

The target is `application_name`. Classes with fewer than 100 training examples are removed to avoid meaningless per-class evaluation on extremely rare labels. This filtering is dataset-specific and is not part of the original paper.


In [4]:
DATA_ROOT="/kaggle/input"
files=[]
for root,dirs,fs in os.walk(DATA_ROOT):
    for f in fs:
        if f.endswith(".parquet"):
            files.append(os.path.join(root,f))
print("\n".join(files))

DATA_FILES=[p for p in files if p.endswith("session1_flows.parquet") or p.endswith("session2_flows.parquet")]
if not DATA_FILES:
    DATA_FILES=files
if not DATA_FILES:
    raise FileNotFoundError("No parquet files found under /kaggle/input")

df=pd.concat([pd.read_parquet(p) for p in DATA_FILES],ignore_index=True)
print(df.shape)
print(df["application_name"].value_counts().head(20))


/kaggle/input/datasets/<kaggle-user>/dataset/VPN-nonVPN-Dataset/Data/session1/session1_flows.parquet
/kaggle/input/datasets/<kaggle-user>/dataset/VPN-nonVPN-Dataset/Data/session1/session1_nfstream_inner_flows.parquet
/kaggle/input/datasets/<kaggle-user>/dataset/VPN-nonVPN-Dataset/Data/session1/session1_packet_matches.parquet
/kaggle/input/datasets/<kaggle-user>/dataset/VPN-nonVPN-Dataset/Data/session2/session2_flows.parquet
/kaggle/input/datasets/<kaggle-user>/dataset/VPN-nonVPN-Dataset/Data/session2/session2_nfstream_inner_flows.parquet
/kaggle/input/datasets/<kaggle-user>/dataset/VPN-nonVPN-Dataset/Data/session2/session2_packet_matches.parquet
(226454, 126)
application_name
DNS                   83541
TLS                   75646
QUIC                  29074
Unknown                5680
HTTP                   4105
TLS.MS_OneDrive        3289
TLS.Facebook           3198
TLS.Microsoft365       2479
TLS.Teams              2100
WhatsApp               1803
Discord                1797
Telegra

In [9]:
TARGET="application_name"
MIN_CLASS_COUNT=100
counts=df[TARGET].value_counts()
valid=counts[counts>=MIN_CLASS_COUNT].index
df=df[df[TARGET].isin(valid)].reset_index(drop=True)
print(df.shape)
print("classes",df[TARGET].nunique())


(225180, 126)
classes 35


In [8]:
EXCLUDED={
"id","expiration_id","src_ip","src_mac","src_oui","src_port","dst_ip","dst_mac","dst_oui","dst_port",
"application_name","application_category_name","application_is_guessed","application_confidence",
"requested_server_name","detected_protocol","flow_id","flow_start_ms","flow_end_ms",
"src2dst_key","dst2src_key","bidirectional_first_seen_ms","bidirectional_last_seen_ms",
"src2dst_first_seen_ms","src2dst_last_seen_ms","dst2src_first_seen_ms","dst2src_last_seen_ms"
}

outer_numeric=[]
for c in df.columns:
    if c in EXCLUDED or not c.startswith("outer_"):
        continue
    if c in {"outer_splt_direction","outer_splt_ps","outer_splt_piat_ms"}:
        continue
    if pd.api.types.is_numeric_dtype(df[c]):
        outer_numeric.append(c)

print("encrypted numeric features",len(outer_numeric))
print(outer_numeric)


encrypted numeric features 21
['outer_bytes', 'outer_duration_ms', 'outer_first_matched_time_ms', 'outer_last_matched_time_ms', 'outer_capture_duration_ms', 'outer_packet_rate', 'outer_byte_rate', 'outer_bytes_in', 'outer_bytes_out', 'outer_min_piat_ms_in', 'outer_mean_piat_ms_in', 'outer_stddev_piat_ms_in', 'outer_max_piat_ms_in', 'outer_min_piat_ms_out', 'outer_mean_piat_ms_out', 'outer_stddev_piat_ms_out', 'outer_max_piat_ms_out', 'outer_min_piat_ms', 'outer_mean_piat_ms', 'outer_stddev_piat_ms', 'outer_max_piat_ms']


In [7]:
def parse_sequence(v):
    if isinstance(v,np.ndarray):
        a=v.tolist()
    elif isinstance(v,list):
        a=v
    elif isinstance(v,str):
        try:
            a=ast.literal_eval(v)
        except Exception:
            a=v.strip("[]")
            a=[] if not a else a.split(",")
    else:
        return np.empty(0,dtype=np.float32)
    out=[]
    for x in a:
        try:
            z=float(x)
            if z>=0:
                out.append(z)
        except Exception:
            pass
    return np.asarray(out,dtype=np.float32)


## 70:30 split

The paper uses a random 70:30 train/test split. The scaler is fitted only on the training partition.


In [12]:
train_df,test_df=train_test_split(df,test_size=0.30,random_state=SEED,stratify=df[TARGET])

le=LabelEncoder()
y_train=le.fit_transform(train_df[TARGET])
y_test=le.transform(test_df[TARGET])
NUM_CLASSES=len(le.classes_)

scaler=StandardScaler()
X_train=scaler.fit_transform(train_df[outer_numeric].astype(np.float32))
X_test=scaler.transform(test_df[outer_numeric].astype(np.float32))

print(len(train_df),len(test_df),NUM_CLASSES,X_train.shape[1])


157626 67554 35 21


## Feature map

The paper reshapes selected flow features into feature maps before the convolutional extractor. Because the new dataset has a different feature count, the vector is zero-padded to the next square size and reshaped into a one-channel 2D map. The convolutional architecture itself remains the paper architecture.


In [13]:
FEATURE_DIM=X_train.shape[1]
GRID_SIDE=int(math.ceil(math.sqrt(FEATURE_DIM)))
GRID_SIZE=GRID_SIDE*GRID_SIDE

def make_map(X,side=GRID_SIDE):
    out=np.zeros((len(X),side*side),dtype=np.float32)
    out[:,:X.shape[1]]=X
    return out.reshape(len(X),1,side,side)

X_train_map=make_map(X_train)
X_test_map=make_map(X_test)
print(X_train_map.shape)


(157626, 1, 5, 5)


In [14]:
class FlowDataset(Dataset):
    def __init__(self,X,y):
        self.X=torch.tensor(X,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self,i):
        return self.X[i],self.y[i]

BATCH_SIZE=256
train_dl=DataLoader(FlowDataset(X_train_map,y_train),batch_size=BATCH_SIZE,shuffle=True,num_workers=0)
test_dl=DataLoader(FlowDataset(X_test_map,y_test),batch_size=BATCH_SIZE,shuffle=False,num_workers=0)


In [15]:
class PaperConvEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(1,32,5,padding=2)
        self.conv2=nn.Conv2d(32,32,5,padding=2)
        self.conv3=nn.Conv2d(32,64,3,padding=1)
        self.conv4=nn.Conv2d(64,64,3,padding=1)
    def forward(self,x):
        x=F.relu(self.conv1(x))
        x=F.relu(self.conv2(x))
        x3=F.relu(self.conv3(x))
        x4=F.relu(self.conv4(x3))
        return torch.cat([x3,x4],dim=1)


In [16]:
class IBLayer(nn.Module):
    def __init__(self,in_dim,z_dim):
        super().__init__()
        self.mu=nn.Linear(in_dim,z_dim)
        self.logvar=nn.Linear(in_dim,z_dim)
    def forward(self,x,num_samples=1):
        mu=self.mu(x)
        logvar=torch.clamp(self.logvar(x),-10.0,10.0)
        var=torch.exp(logvar)
        std=torch.exp(0.5*logvar)
        if self.training or num_samples>1:
            if num_samples==1:
                z=mu+std*torch.randn_like(std)
            else:
                eps=torch.randn(num_samples,*mu.shape,device=mu.device)
                z=(mu.unsqueeze(0)+eps*std.unsqueeze(0)).mean(0)
        else:
            z=mu
        mu_i=mu.unsqueeze(1)
        mu_j=mu.unsqueeze(0)
        var_i=var.unsqueeze(1)
        var_j=var.unsqueeze(0)
        logvar_i=logvar.unsqueeze(1)
        logvar_j=logvar.unsqueeze(0)
        kl=0.5*(logvar_j-logvar_i+(var_i+(mu_i-mu_j).pow(2))/var_j-1.0)
        kl=kl.sum(-1)
        b=x.size(0)
        if b==1:
            mi=torch.zeros((),device=x.device)
        else:
            mi=-(torch.logsumexp(-kl,dim=1)-math.log(b)).mean()
        return z,mi


In [17]:
class IBNN(nn.Module):
    def __init__(self,num_classes,z_dim=128,dropout=0.1):
        super().__init__()
        self.encoder=PaperConvEncoder()
        with torch.no_grad():
            d=self.encoder(torch.zeros(1,1,GRID_SIDE,GRID_SIDE)).flatten(1).shape[1]
        self.extract=nn.Sequential(nn.Linear(d,128),nn.ReLU(True),nn.Dropout(dropout))
        self.ib=IBLayer(128,z_dim)
        self.classifier=nn.Sequential(
            nn.Linear(z_dim,128),nn.ReLU(True),nn.Dropout(dropout),
            nn.Linear(128,256),nn.ReLU(True),nn.Dropout(dropout),
            nn.Linear(256,num_classes)
        )
    def forward(self,x,num_samples=1):
        x=self.encoder(x).flatten(1)
        x=self.extract(x)
        z,mi=self.ib(x,num_samples)
        return self.classifier(z),mi

class NoIBNN(nn.Module):
    def __init__(self,num_classes,dropout=0.1):
        super().__init__()
        self.encoder=PaperConvEncoder()
        with torch.no_grad():
            d=self.encoder(torch.zeros(1,1,GRID_SIDE,GRID_SIDE)).flatten(1).shape[1]
        self.extract=nn.Sequential(nn.Linear(d,128),nn.ReLU(True),nn.Dropout(dropout))
        self.classifier=nn.Sequential(
            nn.Linear(128,128),nn.ReLU(True),nn.Dropout(dropout),
            nn.Linear(128,256),nn.ReLU(True),nn.Dropout(dropout),
            nn.Linear(256,num_classes)
        )
    def forward(self,x):
        x=self.encoder(x).flatten(1)
        x=self.extract(x)
        return self.classifier(x)

ibnn=IBNN(NUM_CLASSES).to(DEVICE)
baseline=NoIBNN(NUM_CLASSES).to(DEVICE)
print("IBNN",sum(p.numel() for p in ibnn.parameters()))
print("No-IBNN",sum(p.numel() for p in baseline.parameters()))


IBNN 583171
No-IBNN 550147


## Training, beta sweep, Fuzzy-GCD, and final evaluation


In [18]:
EPOCHS=30
LR=1e-3
LAMBDA_L2=1e-4
DROPOUT=0.1
Z_DIM=128

BETAS=[0.01,0.005,0.0001,0.00005,0.000001]

def l2(model):
    return sum(torch.sum(p.pow(2)) for p in model.parameters() if p.requires_grad)

def eval_ib(model,dl):
    model.eval()
    yt=[]
    yp=[]
    with torch.no_grad():
        for x,y in dl:
            logits,_=model(x.to(DEVICE),1)
            yp.append(logits.argmax(1).cpu().numpy())
            yt.append(y.numpy())
    return np.concatenate(yt),np.concatenate(yp)

def eval_base(model,dl):
    model.eval()
    yt=[]
    yp=[]
    with torch.no_grad():
        for x,y in dl:
            logits=model(x.to(DEVICE))
            yp.append(logits.argmax(1).cpu().numpy())
            yt.append(y.numpy())
    return np.concatenate(yt),np.concatenate(yp)

def metric_dict(y,p):
    return {
        "accuracy":accuracy_score(y,p),
        "precision_macro":precision_score(y,p,average="macro",zero_division=0),
        "recall_macro":recall_score(y,p,average="macro",zero_division=0),
        "f1_macro":f1_score(y,p,average="macro",zero_division=0)
    }

print("Betas:",BETAS)
print("EPOCHS:",EPOCHS)


Betas: [0.01, 0.005, 0.0001, 5e-05, 1e-06]
EPOCHS: 30


In [20]:
def train_ib(model,dl,beta,desc="IBNN"):
    opt=optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=0.0
    )

    sch=optim.lr_scheduler.OneCycleLR(
        opt,
        max_lr=LR,
        epochs=EPOCHS,
        steps_per_epoch=len(dl)
    )

    hist=[]

    for epoch in range(1,EPOCHS+1):
        model.train()
        total=0.0
        ce_total=0.0
        mi_total=0.0
        seen=0

        bar=tqdm(
            dl,
            desc=f"{desc} Epoch {epoch}/{EPOCHS}"
        )

        for x,y in bar:
            x=x.to(DEVICE)
            y=y.to(DEVICE)

            opt.zero_grad()

            logits,mi=model(x)

            ce=F.cross_entropy(logits,y)
            loss=ce+beta*mi+LAMBDA_L2*l2(model)

            loss.backward()
            opt.step()
            sch.step()

            n=y.size(0)
            seen+=n
            total+=loss.item()*n
            ce_total+=ce.item()*n
            mi_total+=mi.item()*n

            bar.set_postfix(
                loss=f"{loss.item():.4f}",
                ce=f"{ce.item():.4f}",
                mi=f"{mi.item():.4f}"
            )

        yt,yp=eval_ib(model,dl)
        m=metric_dict(yt,yp)

        r={
            "epoch":epoch,
            "loss":total/seen,
            "ce":ce_total/seen,
            "mi":mi_total/seen,
            **m
        }

        hist.append(r)

        print(
            f"{desc} Epoch {epoch}/{EPOCHS} | "
            f"Loss={r['loss']:.4f} | "
            f"CE={r['ce']:.4f} | "
            f"MI={r['mi']:.4f} | "
            f"Acc={r['accuracy']:.4f} | "
            f"Macro-F1={r['f1_macro']:.4f}"
        )

    return hist


In [21]:
ibnn_beta_models={}
ibnn_beta_histories={}
ibnn_beta_results=[]

for beta in BETAS:
    print("\n"+"="*80)
    print(f"TRAINING IBNN WITH BETA = {beta}")
    print("="*80)

    model=IBNN(
        NUM_CLASSES,
        z_dim=Z_DIM,
        dropout=DROPOUT
    ).to(DEVICE)

    history=train_ib(
        model,
        train_dl,
        beta,
        desc=f"IBNN beta={beta}"
    )

    ibnn_beta_histories[beta]=history

    best=max(
        history,
        key=lambda r:r["f1_macro"]
    )

    final=history[-1]

    ibnn_beta_results.append({
        "beta":beta,
        "best_epoch":best["epoch"],
        "best_accuracy":best["accuracy"],
        "best_macro_f1":best["f1_macro"],
        "final_accuracy":final["accuracy"],
        "final_macro_f1":final["f1_macro"]
    })

    ibnn_beta_models[beta]=model

    torch.cuda.empty_cache()

ibnn_beta_df=pd.DataFrame(ibnn_beta_results).sort_values(
    "best_macro_f1",
    ascending=False
).reset_index(drop=True)

display(ibnn_beta_df)



TRAINING IBNN WITH BETA = 0.01


IBNN beta=0.01 Epoch 1/30:   0%|          | 0/616 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [22]:
BEST_BETA=float(ibnn_beta_df.iloc[0]["beta"])

ibnn=ibnn_beta_models[BEST_BETA]

print("Best IBNN beta:",BEST_BETA)
print("IBNN parameters:",f"{sum(p.numel() for p in ibnn.parameters()):,}")

ib_y,ib_p=eval_ib(ibnn,test_dl)
ib_final_metrics=metric_dict(ib_y,ib_p)

print(
    f"IBNN test | "
    f"Acc={ib_final_metrics['accuracy']:.4f} | "
    f"Macro-F1={ib_final_metrics['f1_macro']:.4f}"
)


NameError: name 'ibnn_beta_df' is not defined

In [ ]:
baseline=NoIBNN(
    NUM_CLASSES,
    dropout=DROPOUT
).to(DEVICE)

def train_base(model):
    opt=optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=0.0
    )

    sch=optim.lr_scheduler.OneCycleLR(
        opt,
        max_lr=LR,
        epochs=EPOCHS,
        steps_per_epoch=len(train_dl)
    )

    hist=[]

    for epoch in range(1,EPOCHS+1):
        model.train()
        total=0.0
        seen=0

        bar=tqdm(
            train_dl,
            desc=f"No-IBNN Epoch {epoch}/{EPOCHS}"
        )

        for x,y in bar:
            x=x.to(DEVICE)
            y=y.to(DEVICE)

            opt.zero_grad()

            logits=model(x)

            loss=F.cross_entropy(logits,y)+LAMBDA_L2*l2(model)

            loss.backward()
            opt.step()
            sch.step()

            n=y.size(0)
            seen+=n
            total+=loss.item()*n

            bar.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        yt,yp=eval_base(model,test_dl)
        m=metric_dict(yt,yp)

        r={
            "epoch":epoch,
            "loss":total/seen,
            **m
        }

        hist.append(r)

        print(
            f"No-IBNN Epoch {epoch}/{EPOCHS} | "
            f"Loss={r['loss']:.4f} | "
            f"Acc={r['accuracy']:.4f} | "
            f"Macro-F1={r['f1_macro']:.4f}"
        )

    return hist

baseline_history=train_base(baseline)

base_y,base_p=eval_base(baseline,test_dl)
base_final_metrics=metric_dict(base_y,base_p)

print(
    f"Final No-IBNN test | "
    f"Acc={base_final_metrics['accuracy']:.4f} | "
    f"Macro-F1={base_final_metrics['f1_macro']:.4f}"
)


In [32]:
def fuzzy_gcd_features(sizes,k_min=8,k_max=256,tolerance=4):
    sizes=np.asarray(sizes,dtype=np.float32)
    sizes=sizes[np.isfinite(sizes)&(sizes>0)]

    if len(sizes)==0:
        return np.zeros(4,dtype=np.float32)

    best_k=8.0
    best_score=float("inf")

    for k in range(k_min,k_max+1):
        mult=np.maximum(1.0,np.rint(sizes/k))
        res=np.abs(sizes-mult*k)
        score=res.mean()

        if score<best_score:
            best_score=score
            best_k=float(k)

    mult=np.maximum(1.0,np.rint(sizes/best_k))
    res=np.abs(sizes-mult*best_k)

    return np.array([
        best_k,
        res.mean(),
        np.mean(res/best_k),
        np.mean(res<=tolerance)
    ],dtype=np.float32)

train_seq=train_df["outer_splt_ps"].map(parse_sequence).tolist()
test_seq=test_df["outer_splt_ps"].map(parse_sequence).tolist()

gcd_train=np.vstack([
    fuzzy_gcd_features(s)
    for s in tqdm(train_seq,desc="Fuzzy-GCD train")
])

gcd_test=np.vstack([
    fuzzy_gcd_features(s)
    for s in tqdm(test_seq,desc="Fuzzy-GCD test")
])

print("GCD feature shape:",gcd_train.shape)
print("First GCD features:")
print(gcd_train[:5])


KeyboardInterrupt: 

In [ ]:
X_train_gcd=X_train.copy()
X_test_gcd=X_test.copy()

replace_idx=np.argsort(
    np.var(X_train,axis=0)
)[:4]

print("Replaced feature indices:",replace_idx)

gcd_scaler=StandardScaler()

gcd_train_scaled=gcd_scaler.fit_transform(gcd_train)
gcd_test_scaled=gcd_scaler.transform(gcd_test)

X_train_gcd[:,replace_idx]=gcd_train_scaled
X_test_gcd[:,replace_idx]=gcd_test_scaled

X_train_gcd_map=make_map(X_train_gcd)
X_test_gcd_map=make_map(X_test_gcd)

gcd_train_dl=DataLoader(
    FlowDataset(X_train_gcd_map,y_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

gcd_test_dl=DataLoader(
    FlowDataset(X_test_gcd_map,y_test),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("GCD input shape:",X_train_gcd_map.shape)


In [ ]:
gcd_model=NoIBNN(
    NUM_CLASSES,
    dropout=DROPOUT
).to(DEVICE)

def train_gcd(model):
    opt=optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=0.0
    )

    sch=optim.lr_scheduler.OneCycleLR(
        opt,
        max_lr=LR,
        epochs=EPOCHS,
        steps_per_epoch=len(gcd_train_dl)
    )

    hist=[]

    for epoch in range(1,EPOCHS+1):
        model.train()
        total=0.0
        seen=0

        bar=tqdm(
            gcd_train_dl,
            desc=f"Fuzzy-GCD Epoch {epoch}/{EPOCHS}"
        )

        for x,y in bar:
            x=x.to(DEVICE)
            y=y.to(DEVICE)

            opt.zero_grad()

            logits=model(x)

            loss=F.cross_entropy(logits,y)+LAMBDA_L2*l2(model)

            loss.backward()
            opt.step()
            sch.step()

            n=y.size(0)
            seen+=n
            total+=loss.item()*n

            bar.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        yt,yp=eval_base(model,gcd_test_dl)
        m=metric_dict(yt,yp)

        r={
            "epoch":epoch,
            "loss":total/seen,
            **m
        }

        hist.append(r)

        print(
            f"Fuzzy-GCD Epoch {epoch}/{EPOCHS} | "
            f"Loss={r['loss']:.4f} | "
            f"Acc={r['accuracy']:.4f} | "
            f"Macro-F1={r['f1_macro']:.4f}"
        )

    return hist

gcd_history=train_gcd(gcd_model)

gcd_y,gcd_p=eval_base(gcd_model,gcd_test_dl)
gcd_final_metrics=metric_dict(gcd_y,gcd_p)

print(
    f"Final Fuzzy-GCD test | "
    f"Acc={gcd_final_metrics['accuracy']:.4f} | "
    f"Macro-F1={gcd_final_metrics['f1_macro']:.4f}"
)


In [ ]:
fuzzy_ib_beta_models={}
fuzzy_ib_beta_histories={}
fuzzy_ib_beta_results=[]

for beta in BETAS:
    print("\n"+"="*80)
    print(f"TRAINING FUZZY-GCD + IBNN WITH BETA = {beta}")
    print("="*80)

    model=IBNN(
        NUM_CLASSES,
        z_dim=Z_DIM,
        dropout=DROPOUT
    ).to(DEVICE)

    history=train_ib(
        model,
        gcd_train_dl,
        beta,
        desc=f"Fuzzy-GCD + IBNN beta={beta}"
    )

    fuzzy_ib_beta_histories[beta]=history

    best=max(
        history,
        key=lambda r:r["f1_macro"]
    )

    final=history[-1]

    fuzzy_ib_beta_results.append({
        "beta":beta,
        "best_epoch":best["epoch"],
        "best_accuracy":best["accuracy"],
        "best_macro_f1":best["f1_macro"],
        "final_accuracy":final["accuracy"],
        "final_macro_f1":final["f1_macro"]
    })

    fuzzy_ib_beta_models[beta]=model

    torch.cuda.empty_cache()

fuzzy_ib_beta_df=pd.DataFrame(
    fuzzy_ib_beta_results
).sort_values(
    "best_macro_f1",
    ascending=False
).reset_index(drop=True)

display(fuzzy_ib_beta_df)


In [ ]:
BEST_FUZZY_IB_BETA=float(
    fuzzy_ib_beta_df.iloc[0]["beta"]
)

fuzzy_ibnn=fuzzy_ib_beta_models[BEST_FUZZY_IB_BETA]

print(
    "Best Fuzzy-GCD + IBNN beta:",
    BEST_FUZZY_IB_BETA
)

print(
    "Fuzzy-GCD + IBNN parameters:",
    f"{sum(p.numel() for p in fuzzy_ibnn.parameters()):,}"
)

fuzzy_ib_y,fuzzy_ib_p=eval_ib(
    fuzzy_ibnn,
    gcd_test_dl
)

fuzzy_ib_final_metrics=metric_dict(
    fuzzy_ib_y,
    fuzzy_ib_p
)

print(
    f"Final Fuzzy-GCD + IBNN test | "
    f"Acc={fuzzy_ib_final_metrics['accuracy']:.4f} | "
    f"Macro-F1={fuzzy_ib_final_metrics['f1_macro']:.4f}"
)


In [ ]:
results=pd.DataFrame([
    {
        "Model":"No-IBNN",
        **base_final_metrics
    },
    {
        "Model":"IBNN",
        **ib_final_metrics
    },
    {
        "Model":"Fuzzy-GCD",
        **gcd_final_metrics
    },
    {
        "Model":"Fuzzy-GCD + IBNN",
        **fuzzy_ib_final_metrics
    }
])

results


In [ ]:
print("No-IBNN")
print(
    classification_report(
        base_y,
        base_p,
        target_names=le.classes_,
        zero_division=0
    )
)

print("IBNN")
print(
    classification_report(
        ib_y,
        ib_p,
        target_names=le.classes_,
        zero_division=0
    )
)

print("Fuzzy-GCD")
print(
    classification_report(
        gcd_y,
        gcd_p,
        target_names=le.classes_,
        zero_division=0
    )
)

print("Fuzzy-GCD + IBNN")
print(
    classification_report(
        fuzzy_ib_y,
        fuzzy_ib_p,
        target_names=le.classes_,
        zero_division=0
    )
)


In [ ]:
plt.figure(figsize=(9,6))

plt.plot(
    pd.DataFrame(ibnn_beta_histories[BEST_BETA]).epoch,
    pd.DataFrame(ibnn_beta_histories[BEST_BETA]).f1_macro,
    label=f"IBNN beta={BEST_BETA}"
)

plt.plot(
    pd.DataFrame(baseline_history).epoch,
    pd.DataFrame(baseline_history).f1_macro,
    label="No-IBNN"
)

plt.plot(
    pd.DataFrame(gcd_history).epoch,
    pd.DataFrame(gcd_history).f1_macro,
    label="Fuzzy-GCD"
)

plt.plot(
    pd.DataFrame(fuzzy_ib_beta_histories[BEST_FUZZY_IB_BETA]).epoch,
    pd.DataFrame(fuzzy_ib_beta_histories[BEST_FUZZY_IB_BETA]).f1_macro,
    label=f"Fuzzy-GCD + IBNN beta={BEST_FUZZY_IB_BETA}"
)

plt.xlabel("Epoch")
plt.ylabel("Macro F1")
plt.title("Macro F1 Comparison")
plt.grid(True,alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [4]:
os.makedirs("/kaggle/working/models",exist_ok=True)

torch.save(
    baseline.state_dict(),
    "/kaggle/working/models/no_ibnn.pt"
)

torch.save(
    ibnn.state_dict(),
    "/kaggle/working/models/ibnn.pt"
)

torch.save(
    gcd_model.state_dict(),
    "/kaggle/working/models/fuzzy_gcd.pt"
)

torch.save(
    fuzzy_ibnn.state_dict(),
    "/kaggle/working/models/fuzzy_gcd_ibnn.pt"
)

import joblib

joblib.dump(
    scaler,
    "/kaggle/working/models/feature_scaler.pkl"
)

joblib.dump(
    gcd_scaler,
    "/kaggle/working/models/gcd_scaler.pkl"
)

joblib.dump(
    le,
    "/kaggle/working/models/label_encoder.pkl"
)

np.save(
    "/kaggle/working/models/feature_names.npy",
    np.array(outer_numeric,dtype=object)
)

config={
    "num_classes":int(NUM_CLASSES),
    "classes":le.classes_.tolist(),
    "feature_dim":int(FEATURE_DIM),
    "grid_side":int(GRID_SIDE),
    "batch_size":int(BATCH_SIZE),
    "epochs":int(EPOCHS),
    "lr":float(LR),
    "lambda_l2":float(LAMBDA_L2),
    "best_beta":float(BEST_BETA),
    "best_fuzzy_ib_beta":float(BEST_FUZZY_IB_BETA),
    "z_dim":int(Z_DIM),
    "dropout":float(DROPOUT),
    "gcd_replace_indices":replace_idx.tolist()
}

with open(
    "/kaggle/working/models/config.json",
    "w"
) as f:
    import json
    json.dump(config,f,indent=2)

print("Saved models and preprocessing artifacts to /kaggle/working/models")


NameError: name 'baseline' is not defined

In [27]:
import os
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

MODEL_DIR = "/kaggle/working/models"

with open(os.path.join(MODEL_DIR, "label_encoder.pkl"), "rb") as f:
    le_saved = pickle.load(f)

with open(os.path.join(MODEL_DIR, "feature_scaler.pkl"), "rb") as f:
    feature_scaler_saved = pickle.load(f)

with open(os.path.join(MODEL_DIR, "gcd_scaler.pkl"), "rb") as f:
    gcd_scaler_saved = pickle.load(f)

feature_names_saved = np.load(
    os.path.join(MODEL_DIR, "feature_names.npy"),
    allow_pickle=True
)

class_names = le_saved.classes_
NUM_CLASSES = len(class_names)

print("Classes:", NUM_CLASSES)
print("Feature count:", len(feature_names_saved))
print(class_names)

UnpicklingError: STACK_GLOBAL requires str

In [24]:
from sklearn.metrics import f1_score
import pandas as pd
import matplotlib.pyplot as plt

models_and_loaders = {
    "No-IBNN": (baseline, test_dl, eval_base),
    "IBNN": (ibnn, test_dl, eval_ib),
    "Fuzzy-GCD": (gcd_model, gcd_test_dl, eval_base),
    "Fuzzy-GCD + IBNN": (fuzzy_ibnn, gcd_test_dl, eval_ib)
}

weighted_f1 = {}

for name, (model, loader, eval_fn) in models_and_loaders.items():
    y_true, y_pred = eval_fn(model, loader)
    weighted_f1[name] = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

weighted_f1_df = pd.DataFrame(
    list(weighted_f1.items()),
    columns=["Model", "Weighted F1"]
)

display(weighted_f1_df)

plt.figure(figsize=(9, 6))

plt.bar(
    weighted_f1_df["Model"],
    weighted_f1_df["Weighted F1"]
)

plt.ylabel("Weighted F1")
plt.xlabel("Model")
plt.title("Support-Weighted F1 Comparison")
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.3)

for i, v in enumerate(weighted_f1_df["Weighted F1"]):
    plt.text(i, v + 0.01, f"{v:.4f}", ha="center")

plt.tight_layout()
plt.show()

NameError: name 'gcd_model' is not defined

In [33]:
import os
import numpy as np
from tqdm.auto import tqdm

MODEL_DIR = "/kaggle/working/models"

test_seq = test_df["outer_splt_ps"].map(parse_sequence).tolist()

gcd_test = np.vstack([
    fuzzy_gcd_features(s)
    for s in tqdm(test_seq, desc="Computing GCD features")
])

gcd_test_scaled = gcd_scaler.transform(gcd_test)

np.save(
    os.path.join(MODEL_DIR, "gcd_test.npy"),
    gcd_test
)

np.save(
    os.path.join(MODEL_DIR, "gcd_test_scaled.npy"),
    gcd_test_scaled
)

X_test_gcd = X_test.copy()
X_test_gcd[:, replace_idx] = gcd_test_scaled

X_test_gcd_map = make_map(X_test_gcd)

np.save(
    os.path.join(MODEL_DIR, "X_test_gcd.npy"),
    X_test_gcd
)

np.save(
    os.path.join(MODEL_DIR, "X_test_gcd_map.npy"),
    X_test_gcd_map
)

np.save(
    os.path.join(MODEL_DIR, "replace_idx.npy"),
    replace_idx
)

print("GCD features saved.")
print("gcd_test:", gcd_test.shape)
print("gcd_test_scaled:", gcd_test_scaled.shape)
print("X_test_gcd_map:", X_test_gcd_map.shape)

Computing GCD features:   0%|          | 0/67554 [00:00<?, ?it/s]

NameError: name 'gcd_scaler' is not defined

In [34]:
import os
import joblib
import numpy as np

MODEL_DIR = "/kaggle/working/models"

gcd_scaler = joblib.load(
    os.path.join(MODEL_DIR, "gcd_scaler.pkl")
)

gcd_test_scaled = gcd_scaler.transform(gcd_test)

print(gcd_test_scaled.shape)

(67554, 4)


In [37]:
import os
import numpy as np
import joblib

MODEL_DIR="/kaggle/working/models"

replace_idx=np.argsort(np.var(X_train,axis=0))[:4]

gcd_scaler=joblib.load(
    os.path.join(MODEL_DIR,"gcd_scaler.pkl")
)

gcd_test_scaled=gcd_scaler.transform(gcd_test)

X_test_gcd=X_test.copy()
X_test_gcd[:,replace_idx]=gcd_test_scaled

X_test_gcd_map=make_map(X_test_gcd)

np.save(os.path.join(MODEL_DIR,"gcd_test.npy"),gcd_test)
np.save(os.path.join(MODEL_DIR,"gcd_test_scaled.npy"),gcd_test_scaled)
np.save(os.path.join(MODEL_DIR,"X_test_gcd.npy"),X_test_gcd)
np.save(os.path.join(MODEL_DIR,"X_test_gcd_map.npy"),X_test_gcd_map)
np.save(os.path.join(MODEL_DIR,"replace_idx.npy"),replace_idx)

print("GCD features saved")
print("replace_idx:",replace_idx)
print("gcd_test:",gcd_test.shape)
print("X_test_gcd_map:",X_test_gcd_map.shape)

GCD features saved
replace_idx: [5 2 7 1]
gcd_test: (67554, 4)
X_test_gcd_map: (67554, 1, 5, 5)


In [38]:
X_test_gcd=X_test.copy()

X_test_gcd[:,replace_idx]=gcd_test_scaled

X_test_gcd_map=make_map(X_test_gcd)

np.save(
    os.path.join(MODEL_DIR,"gcd_test.npy"),
    gcd_test
)

np.save(
    os.path.join(MODEL_DIR,"gcd_test_scaled.npy"),
    gcd_test_scaled
)

np.save(
    os.path.join(MODEL_DIR,"X_test_gcd.npy"),
    X_test_gcd
)

np.save(
    os.path.join(MODEL_DIR,"X_test_gcd_map.npy"),
    X_test_gcd_map
)

print("GCD test features saved.")

GCD test features saved.


In [40]:
import os
import numpy as np
import torch

MODEL_DIR="/kaggle/working/models"

class_names=le.classes_

baseline=NoIBNN(len(class_names)).to(DEVICE)
ibnn=IBNN(len(class_names)).to(DEVICE)

fuzzy_gcd_model=NoIBNN(len(class_names)).to(DEVICE)
fuzzy_gcd_ibnn_model=IBNN(len(class_names)).to(DEVICE)

baseline.load_state_dict(
    torch.load(
        os.path.join(MODEL_DIR,"no_ibnn.pt"),
        map_location=DEVICE
    )
)

ibnn.load_state_dict(
    torch.load(
        os.path.join(MODEL_DIR,"ibnn.pt"),
        map_location=DEVICE
    )
)

fuzzy_gcd_model.load_state_dict(
    torch.load(
        os.path.join(MODEL_DIR,"fuzzy_gcd.pt"),
        map_location=DEVICE
    )
)

fuzzy_gcd_ibnn_model.load_state_dict(
    torch.load(
        os.path.join(MODEL_DIR,"fuzzy_gcd_ibnn.pt"),
        map_location=DEVICE
    )
)

baseline.eval()
ibnn.eval()
fuzzy_gcd_model.eval()
fuzzy_gcd_ibnn_model.eval()

print("All four models loaded.")

All four models loaded.


In [41]:
X_test_gcd_map=np.load(
    os.path.join(MODEL_DIR,"X_test_gcd_map.npy")
)

print(X_test_gcd_map.shape)

(67554, 1, 5, 5)


In [44]:
from torch.utils.data import DataLoader

baseline_test_dl=DataLoader(
    FlowDataset(X_test_map,y_test),
    batch_size=256,
    shuffle=False,
    num_workers=0
)

gcd_test_dl=DataLoader(
    FlowDataset(X_test_gcd_map,y_test),
    batch_size=256,
    shuffle=False,
    num_workers=0
)

In [47]:
import os
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader

MODEL_DIR="/kaggle/working/models"
class_names=le.classes_

X_test_gcd_map=np.load(
    os.path.join(MODEL_DIR,"X_test_gcd_map.npy")
)

normal_dl=DataLoader(
    FlowDataset(X_test_map,y_test),
    batch_size=256,
    shuffle=False,
    num_workers=0
)

gcd_dl=DataLoader(
    FlowDataset(X_test_gcd_map,y_test),
    batch_size=256,
    shuffle=False,
    num_workers=0
)

def predict_no_ib(model,loader):
    predictions=[]
    model.eval()

    with torch.no_grad():
        for X,_ in loader:
            X=X.to(DEVICE)
            logits=model(X)
            predictions.append(
                torch.argmax(logits,dim=1).cpu().numpy()
            )

    return np.concatenate(predictions)

def predict_ib(model,loader):
    predictions=[]
    model.eval()

    with torch.no_grad():
        for X,_ in loader:
            X=X.to(DEVICE)
            logits,_=model(X)
            predictions.append(
                torch.argmax(logits,dim=1).cpu().numpy()
            )

    return np.concatenate(predictions)

pred_no_ibnn=predict_no_ib(baseline,normal_dl)
pred_ibnn=predict_ib(ibnn,normal_dl)
pred_fuzzy_gcd=predict_no_ib(fuzzy_gcd_model,gcd_dl)
pred_fuzzy_gcd_ibnn=predict_ib(fuzzy_gcd_ibnn_model,gcd_dl)

labels=np.arange(len(class_names))

cm_no_ibnn=confusion_matrix(
    y_test,pred_no_ibnn,labels=labels
)

cm_ibnn=confusion_matrix(
    y_test,pred_ibnn,labels=labels
)

cm_fuzzy_gcd=confusion_matrix(
    y_test,pred_fuzzy_gcd,labels=labels
)

cm_fuzzy_gcd_ibnn=confusion_matrix(
    y_test,pred_fuzzy_gcd_ibnn,labels=labels
)

def class_accuracy(cm):
    total=cm.sum(axis=1)
    correct=np.diag(cm)

    return np.divide(
        correct,
        total,
        out=np.zeros(len(correct),dtype=float),
        where=total!=0
    )*100

acc_no_ibnn=class_accuracy(cm_no_ibnn)
acc_ibnn=class_accuracy(cm_ibnn)
acc_fuzzy_gcd=class_accuracy(cm_fuzzy_gcd)
acc_fuzzy_gcd_ibnn=class_accuracy(cm_fuzzy_gcd_ibnn)

ranking_df=pd.DataFrame({
    "Class":class_names,
    "No-IBNN Accuracy (%)":acc_no_ibnn,
    "IBNN Accuracy (%)":acc_ibnn,
    "Fuzzy-GCD Accuracy (%)":acc_fuzzy_gcd,
    "Fuzzy-GCD + IBNN Accuracy (%)":acc_fuzzy_gcd_ibnn
})

ranking_df["GCD vs No-IBNN (pp)"]=(
    ranking_df["Fuzzy-GCD Accuracy (%)"]
    -ranking_df["No-IBNN Accuracy (%)"]
)

ranking_df["GCD+IBNN vs IBNN (pp)"]=(
    ranking_df["Fuzzy-GCD + IBNN Accuracy (%)"]
    -ranking_df["IBNN Accuracy (%)"]
)

print("Confusion matrices:")
print("No-IBNN:",cm_no_ibnn.shape)
print("IBNN:",cm_ibnn.shape)
print("Fuzzy-GCD:",cm_fuzzy_gcd.shape)
print("Fuzzy-GCD + IBNN:",cm_fuzzy_gcd_ibnn.shape)

print("\nAll classes ranked by No-IBNN:")
display(
    ranking_df[
        ["Class","No-IBNN Accuracy (%)"]
    ].sort_values(
        "No-IBNN Accuracy (%)",
        ascending=False
    ).reset_index(drop=True)
)

print("\nAll classes ranked by IBNN:")
display(
    ranking_df[
        ["Class","IBNN Accuracy (%)"]
    ].sort_values(
        "IBNN Accuracy (%)",
        ascending=False
    ).reset_index(drop=True)
)

print("\nAll classes ranked by Fuzzy-GCD:")
display(
    ranking_df[
        ["Class","Fuzzy-GCD Accuracy (%)"]
    ].sort_values(
        "Fuzzy-GCD Accuracy (%)",
        ascending=False
    ).reset_index(drop=True)
)

print("\nAll classes ranked by Fuzzy-GCD + IBNN:")
display(
    ranking_df[
        ["Class","Fuzzy-GCD + IBNN Accuracy (%)"]
    ].sort_values(
        "Fuzzy-GCD + IBNN Accuracy (%)",
        ascending=False
    ).reset_index(drop=True)
)

print("\nCombined comparison:")
display(
    ranking_df.sort_values(
        "GCD vs No-IBNN (pp)",
        ascending=False
    ).reset_index(drop=True)
)

Confusion matrices:
No-IBNN: (35, 35)
IBNN: (35, 35)
Fuzzy-GCD: (35, 35)
Fuzzy-GCD + IBNN: (35, 35)

All classes ranked by No-IBNN:


,Class,No-IBNN Accuracy (%)
0,DNS,98.850850
1,TLS,84.286596
2,QUIC,70.534281
3,Unknown,60.739437
4,Discord,27.087199
5,HTTP,22.745735
6,STUN.TeamsCall,17.672414
7,STUN.WhatsAppCall,17.460317
8,TLS.Facebook,14.598540
9,TLS.MS_OneDrive,3.850051



All classes ranked by IBNN:


,Class,IBNN Accuracy (%)
0,DNS,98.854840
1,TLS,84.348286
2,QUIC,79.706489
3,HTTP.UbuntuONE,77.210884
4,Unknown,61.150235
5,Telegram,36.547085
6,Discord,30.983302
7,TLS.MS_OneDrive,26.342452
8,STUN.WhatsAppCall,22.222222
9,STUN.TeamsCall,17.241379



All classes ranked by Fuzzy-GCD:


,Class,Fuzzy-GCD Accuracy (%)
0,DNS,99.118187
1,QUIC,94.966751
2,TLS,91.561646
3,TLS.Facebook,82.690302
4,Unknown,60.035211
5,Telegram,29.147982
6,TLS.MS_OneDrive,25.430598
7,Discord,24.675325
8,HTTP,19.415110
9,STUN.TeamsCall,17.672414



All classes ranked by Fuzzy-GCD + IBNN:


,Class,Fuzzy-GCD + IBNN Accuracy (%)
0,DNS,98.838880
1,QUIC,94.783307
2,TLS,91.588085
3,TLS.Facebook,80.709072
4,Unknown,61.091549
5,HTTP.UbuntuONE,47.619048
6,Telegram,31.838565
7,TLS.MS_OneDrive,25.633232
8,Discord,24.675325
9,HTTP,20.389927



Combined comparison:


,Class,No-IBNN Accuracy (%),IBNN Accuracy (%),Fuzzy-GCD Accuracy (%),Fuzzy-GCD + IBNN Accuracy (%),GCD vs No-IBNN (pp),GCD+IBNN vs IBNN (pp)
0,TLS.Facebook,14.598540,0.521376,82.690302,80.709072,68.091762,80.187696
1,Telegram,0.000000,36.547085,29.147982,31.838565,29.147982,-4.708520
2,QUIC,70.534281,79.706489,94.966751,94.783307,24.432470,15.076817
3,TLS.MS_OneDrive,3.850051,26.342452,25.430598,25.633232,21.580547,-0.709220
4,TLS,84.286596,84.348286,91.561646,91.588085,7.275051,7.239799
5,HTTP.UbuntuONE,0.000000,77.210884,4.761905,47.619048,4.761905,-29.591837
6,TLS.Teams,0.000000,1.746032,2.380952,2.698413,2.380952,0.952381
7,DNS,98.850850,98.854840,99.118187,98.838880,0.267337,-0.015960
8,Cassandra,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9,TLS.Microsoft365,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
